<a href="https://colab.research.google.com/github/demekeendalie/multi-emotion-classification/blob/main/multi_label_emotion_classification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
import torch
from sklearn.model_selection import train_test_split
from transformers import XLMRobertaTokenizer, XLMRobertaForSequenceClassification, Trainer, TrainingArguments
from datasets import load_dataset, Dataset

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import pandas as pd

# Try reading the CSV with different encodings
data = pd.read_excel('/content/drive/MyDrive/preprocessed emotion_dataset.xlsx')
display(data.tail(10))

,comments,sad,happy,anger,fear,disgust,surprise,contempt,neutral
22020,አሁንም ዐቢይ አህመድ ኢትዮጵያን ቅስሟን ሰብሬ እጥላለው እያለ እየታገለ ...,0,0,1,1,1,0,0,0
22021,98 የወለጋ የተፈናቀለው የተገደለው ሙስሊም አማራ ነው ኦሮሞ እና ሙስሊም...,0,0,1,0,1,1,0,0
22022,ጠላት ደነገጠ በባህር ኃይላችን ፐ ፐ ፐ ባህር ኃይል ችግኝ ቤተመንግስት ...,1,0,0,0,0,1,0,0
22023,በዘር ጥላቻ ተወጥረህ መቸም አይገባህም ስለአገር ልክልክህን ነግረውሃ ረጅ...,0,0,1,0,1,0,0,0
22024,እርፍ በል ሽሜ ሊያውም ሙሉ ብሔራዊ ቡድኑን ነው ያደመከው። የሚገርም ህብ...,0,0,1,0,0,0,0,0
22025,ለኔ የምንግዜም ጀግና ሁሌም አምንህ ነበር ሁሉም በአብይ ፍቅር ባበደበት ...,1,0,0,0,0,1,0,0
22026,የአብይ ጥበቃ አራዊት ከብልፅግና በላይ ፅንፈኛ ከብልፅግና በላይ የኢትዮጵ...,0,0,1,0,1,0,0,0
22027,አለምነህ ዋሴ ምርጥ ይሁዳዊ ጋዜጠኛ በርታልን እኔ የኛ ሰው መሆንክን ባወ...,1,0,0,0,0,1,0,0
22028,ሰው ሁሉ እንደ እርስዎ አላዋቂ፣ ግብዝ እና ተግባር በሌለው ተንኮልን ባነ...,0,0,1,0,1,0,0,0
22029,ሶፎንያስ በሕይወት ሲኖር ያማልዳል !!!ከሞት ቦኃላ ወደ እንጦርጦስ ስለሆ...,0,0,0,0,0,0,0,0


In [ ]:
print(data.head)

<bound method NDFrame.head of                                                 comments  sad  happy  anger  \
0           ብአድን ካልጠፍ መቸም አማራ ሰላም ጠረነቱን አያሸንፍም በጣም ያሳዝናል    1      0      1   
1      የአማራ ልጆች መከላከያ አንድገቡ አበረታቱ ሀገር መከላከያ ገብተው ምድር ...    0      0      0   
2                             ፍትህ ለምዕራብ ወሎ ህዝብ በተለይ በገጠሩ    0      0      0   
3      ጠላት መነሀሪያ አድርጎታል እባካችሁ የሜድያ ባለቤቶች እውነት ስበአዋነት ...    1      0      0   
4      ማንው ፍኖን እሚበትነው ፍኖን እበትናለሁ የሚል አመራር የብአድን የብልጽግ...    0      0      1   
...                                                  ...  ...    ...    ...   
22025  ለኔ የምንግዜም ጀግና ሁሌም አምንህ ነበር ሁሉም በአብይ ፍቅር ባበደበት ...    1      0      0   
22026  የአብይ ጥበቃ አራዊት ከብልፅግና በላይ ፅንፈኛ ከብልፅግና በላይ የኢትዮጵ...    0      0      1   
22027  አለምነህ ዋሴ ምርጥ ይሁዳዊ ጋዜጠኛ በርታልን እኔ የኛ ሰው መሆንክን ባወ...    1      0      0   
22028  ሰው ሁሉ እንደ እርስዎ አላዋቂ፣ ግብዝ እና ተግባር በሌለው ተንኮልን ባነ...    0      0      1   
22029  ሶፎንያስ በሕይወት ሲኖር ያማልዳል !!!ከሞት ቦኃላ ወደ እንጦርጦስ ስለሆ...    0      0      0   

       fear  disgust 

In [ ]:
from sklearn.preprocessing import MultiLabelBinarizer

# Extract label columns from the DataFrame
label_columns = ['sad', 'happy', 'anger', 'fear', 'disgust', 'surprise', 'contempt', 'neutral']
labels = data[label_columns]

# The labels are already in binary format, so we can directly use their values.
binary_labels = labels.values

In [ ]:
# train test split
from sklearn.model_selection import train_test_split

# First split: data and corresponding binary_labels
train_val_df, test_dataset, train_val_labels, test_labels = train_test_split(
    data, binary_labels, test_size=0.10, random_state=42
)

# Second split: train_val_df and train_val_labels
train_dataset, evaluation_dataset, train_labels, val_labels = train_test_split(
    train_val_df, train_val_labels, test_size=0.10, random_state=42
)

print('Training dataset shape: ', train_dataset.shape)
print('Validation dataset shape: ', evaluation_dataset.shape)
print('Testing dataset shape: ', test_dataset.shape)

Training dataset shape:  (17844, 9)
Validation dataset shape:  (1983, 9)
Testing dataset shape:  (2203, 9)


In [ ]:
# Initialize tokenizer
tokenizer = XLMRobertaTokenizer.from_pretrained("Davlan/afro-xlmr-base")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/398 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

In [ ]:
# Extract texts from the datasets
train_texts = train_dataset['comments'].tolist()
val_texts = evaluation_dataset['comments'].tolist()
test_texts = test_dataset['comments'].tolist()

# Tokenize the data
train_encodings = tokenizer(train_texts, truncation=True, padding=True, max_length=512)
val_encodings = tokenizer(val_texts, truncation=True, padding=True, max_length=512)
test_encodings=tokenizer(test_texts, truncation=True, padding=True, max_length=512)

In [ ]:
# Create PyTorch Dataset
class MultiLabelDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx], dtype=torch.float)  # Use float for multi-label
        return item

    def __len__(self):
        return len(self.labels)

In [ ]:
train_dataset = MultiLabelDataset(train_encodings, train_labels)
val_dataset = MultiLabelDataset(val_encodings, val_labels)
test_dataset = MultiLabelDataset(test_encodings, test_labels)

In [ ]:
# Load model
model = XLMRobertaForSequenceClassification.from_pretrained(
    "Davlan/afro-xlmr-base",
    num_labels=len(label_columns),
    problem_type="multi_label_classification"  # Set problem type
)

config.json:   0%|          | 0.00/707 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.11G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

XLMRobertaForSequenceClassification LOAD REPORT from: Davlan/afro-xlmr-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
classifier.dense.bias           | MISSING    | 
classifier.out_proj.weight      | MISSING    | 
classifier.dense.weight         | MISSING    | 
classifier.out_proj.bias        | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [ ]:
# Set training arguments
training_args = TrainingArguments(
   output_dir="./multi_emotion",
   eval_strategy='epoch',
   save_strategy='epoch',
   logging_strategy='epoch',
   num_train_epochs=5,
   learning_rate=1e-5,
   per_device_train_batch_size=8,  # batch size per device during training
   per_device_eval_batch_size=8,   # batch size for evaluation
   warmup_steps=1000,                # number of warmup steps for learning rate
   weight_decay=0.02,
   logging_dir='./logs',            # directory for storing logs
   logging_steps=20,
   report_to="none",
   load_best_model_at_end= True,
)

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [ ]:
pip install evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 10.3 MB/s eta 0:00:00


In [ ]:
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    accuracy_score,
    hamming_loss,
    precision_recall_fscore_support
)
import numpy as np
import torch

# Optional: provide label names list, e.g. label_names = ["sports","politics",..."]
label_names = None  # or a list of strings of length num_labels

def custom_metrics(eval_pred):
    logits, labels = eval_pred

    # Convert logits -> probabilities -> binary predictions
    probabilities = torch.sigmoid(torch.from_numpy(logits)).numpy()
    predictions = (probabilities >= 0.5).astype(int)
    references = labels.astype(int)

    # Micro-averaged metrics
    f1_micro = f1_score(references, predictions, average="micro")
    precision_micro = precision_score(references, predictions, average="micro")
    recall_micro = recall_score(references, predictions, average="micro")

    # Macro-averaged metrics
    f1_macro = f1_score(references, predictions, average="macro")
    precision_macro = precision_score(references, predictions, average="macro")
    recall_macro = recall_score(references, predictions, average="macro")

    accuracy = accuracy_score(references, predictions)
    hamming = hamming_loss(references, predictions)

    # Per-label metrics
    per_prec, per_rec, per_f1, per_support = precision_recall_fscore_support(
        references, predictions, average=None, zero_division=0
    )

    num_labels = per_f1.shape[0]
    if label_names is None:
        names = [f"label_{i}" for i in range(num_labels)]
    else:
        names = label_names
        if len(names) != num_labels:
            raise ValueError("label_names length must match number of labels")

    # Print per-label table and summary
    print("\nPer-label classification report:")
    print(f"{'label':30s}{'precision':>10s}{'recall':>10s}{'f1':>10s}{'support':>10s}")
    for i, name in enumerate(names):
        print(f"{name:30s}{per_prec[i]:10.4f}{per_rec[i]:10.4f}{per_f1[i]:10.4f}{int(per_support[i]):10d}")

    # Print averaged metrics
    print("\nAveraged metrics:")
    print(f"{'metric':20s}{'micro':>10s}{'macro':>10s}")
    print(f"{'precision':20s}{precision_micro:10.4f}{precision_macro:10.4f}")
    print(f"{'recall':20s}{recall_micro:10.4f}{recall_macro:10.4f}")
    print(f"{'f1':20s}{f1_micro:10.4f}{f1_macro:10.4f}")
    print(f"{'accuracy':20s}{accuracy:10.4f}")
    print(f"{'hamming_loss':20s}{hamming:10.4f}")

    # Prepare metrics dict for HF Trainer (scalars only)
    metrics = {
        "f1_micro": float(f1_micro),
        "precision_micro": float(precision_micro),
        "recall_micro": float(recall_micro),
        "f1_macro": float(f1_macro),
        "precision_macro": float(precision_macro),
        "recall_macro": float(recall_macro),
        "accuracy": float(accuracy),
        "hamming_loss": float(hamming),
    }

    # Add per-label metrics to metrics dict (optional; Trainer will log scalars)
    for i, name in enumerate(names):
        safe_name = name.replace(" ", "_")
        metrics[f"f1_{safe_name}"] = float(per_f1[i])
        metrics[f"precision_{safe_name}"] = float(per_prec[i])
        metrics[f"recall_{safe_name}"] = float(per_rec[i])
        metrics[f"support_{safe_name}"] = int(per_support[i])

    return metrics

In [ ]:
# Trainer
from transformers import EarlyStoppingCallback
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=custom_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

In [ ]:


# Train the model
trainer.train()

Epoch,Training Loss,Validation Loss,F1 Micro,Precision Micro,Recall Micro,F1 Macro,Precision Macro,Recall Macro,Accuracy,Hamming Loss,F1 Label 0,Precision Label 0,Recall Label 0,Support Label 0,F1 Label 1,Precision Label 1,Recall Label 1,Support Label 1,F1 Label 2,Precision Label 2,Recall Label 2,Support Label 2,F1 Label 3,Precision Label 3,Recall Label 3,Support Label 3,F1 Label 4,Precision Label 4,Recall Label 4,Support Label 4,F1 Label 5,Precision Label 5,Recall Label 5,Support Label 5,F1 Label 6,Precision Label 6,Recall Label 6,Support Label 6,F1 Label 7,Precision Label 7,Recall Label 7,Support Label 7,Runtime,Samples Per Second,Steps Per Second
1,0.377203,0.240003,0.866287,0.944412,0.800100,0.866915,0.946822,0.804797,0.524458,0.093230,0.921594,0.956000,0.889578,806,0.879310,0.927273,0.836066,732,0.746459,0.970534,0.606444,869,0.913183,0.974271,0.859304,661,0.846250,0.897878,0.800236,846,0.911950,0.955519,0.872180,665,0.847844,0.975655,0.749640,695,0.868732,0.917445,0.824930,714,7.101300,279.244000,34.923000
2,0.225431,0.207108,0.883188,0.942592,0.830828,0.883975,0.944085,0.833928,0.586989,0.082955,0.935525,0.953608,0.918114,806,0.887509,0.896792,0.878415,732,0.805007,0.941448,0.703107,869,0.917742,0.982729,0.860817,661,0.866454,0.942976,0.801418,846,0.916078,0.957377,0.878195,665,0.862151,0.966071,0.778417,695,0.881331,0.911677,0.852941,714,7.061900,280.804000,35.118000
3,0.192983,0.195537,0.892891,0.917680,0.869405,0.894665,0.921910,0.869780,0.606657,0.078732,0.930502,0.966578,0.897022,806,0.901798,0.913165,0.890710,732,0.832558,0.841363,0.823936,869,0.921474,0.979557,0.869894,661,0.887059,0.882904,0.891253,846,0.921194,0.937695,0.905263,665,0.869301,0.921095,0.823022,695,0.893431,0.932927,0.857143,714,7.110600,278.879000,34.877000
4,0.171495,0.193797,0.896173,0.921638,0.872077,0.897569,0.924674,0.872577,0.625819,0.076273,0.934252,0.943110,0.925558,806,0.901330,0.923960,0.879781,732,0.838973,0.850888,0.827388,869,0.921850,0.974705,0.874433,661,0.890909,0.914179,0.868794,846,0.917178,0.935837,0.899248,665,0.877489,0.937807,0.824460,695,0.898571,0.916910,0.880952,714,7.100800,279.264000,34.926000
5,0.157198,0.196479,0.895469,0.915547,0.876253,0.896743,0.918089,0.876884,0.622794,0.077219,0.935525,0.953608,0.918114,806,0.901774,0.900545,0.903005,732,0.840936,0.854935,0.827388,869,0.924051,0.968491,0.883510,661,0.890746,0.899879,0.881797,846,0.918755,0.927914,0.909774,665,0.869894,0.917065,0.827338,695,0.892263,0.922272,0.864146,714,7.065400,280.663000,35.101000



Per-label classification report:
label                          precision    recall        f1   support
label_0                           0.9560    0.8896    0.9216       806
label_1                           0.9273    0.8361    0.8793       732
label_2                           0.9705    0.6064    0.7465       869
label_3                           0.9743    0.8593    0.9132       661
label_4                           0.8979    0.8002    0.8462       846
label_5                           0.9555    0.8722    0.9119       665
label_6                           0.9757    0.7496    0.8478       695
label_7                           0.9174    0.8249    0.8687       714

Averaged metrics:
metric                   micro     macro
precision               0.9444    0.9468
recall                  0.8001    0.8048
f1                      0.8663    0.8669
accuracy                0.5245
hamming_loss            0.0932


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Per-label classification report:
label                          precision    recall        f1   support
label_0                           0.9536    0.9181    0.9355       806
label_1                           0.8968    0.8784    0.8875       732
label_2                           0.9414    0.7031    0.8050       869
label_3                           0.9827    0.8608    0.9177       661
label_4                           0.9430    0.8014    0.8665       846
label_5                           0.9574    0.8782    0.9161       665
label_6                           0.9661    0.7784    0.8622       695
label_7                           0.9117    0.8529    0.8813       714

Averaged metrics:
metric                   micro     macro
precision               0.9426    0.9441
recall                  0.8308    0.8339
f1                      0.8832    0.8840
accuracy                0.5870
hamming_loss            0.0830


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Per-label classification report:
label                          precision    recall        f1   support
label_0                           0.9536    0.9181    0.9355       806
label_1                           0.9005    0.9030    0.9018       732
label_2                           0.8549    0.8274    0.8409       869
label_3                           0.9685    0.8835    0.9241       661
label_4                           0.8999    0.8818    0.8907       846
label_5                           0.9279    0.9098    0.9188       665
label_6                           0.9171    0.8273    0.8699       695
label_7                           0.9223    0.8641    0.8923       714

Averaged metrics:
metric                   micro     macro
precision               0.9155    0.9181
recall                  0.8763    0.8769
f1                      0.8955    0.8967
accuracy                0.6228
hamming_loss            0.0772


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye

TrainOutput(global_step=11155, training_loss=0.22486175578146012, metrics={'train_runtime': 1081.3453, 'train_samples_per_second': 82.508, 'train_steps_per_second': 10.316, 'total_flos': 1.173801649102848e+16, 'train_loss': 0.22486175578146012, 'epoch': 5.0})

In [ ]:
test_results = trainer.evaluate(test_dataset)
print(test_results)


Per-label classification report:
label                          precision    recall        f1   support
label_0                           0.9476    0.9314    0.9394       874
label_1                           0.9384    0.8692    0.9025       841
label_2                           0.8240    0.8073    0.8156       934
label_3                           0.9632    0.8625    0.9101       698
label_4                           0.8899    0.8534    0.8713       928
label_5                           0.9300    0.8541    0.8904       747
label_6                           0.9224    0.7873    0.8495       785
label_7                           0.8742    0.8627    0.8684       765

Averaged metrics:
metric                   micro     macro
precision               0.9078    0.9112
recall                  0.8535    0.8535
f1                      0.8798    0.8809
accuracy                0.5847
hamming_loss            0.0870
{'eval_loss': 0.2078186422586441, 'eval_f1_micro': 0.8797741353619324, 'eval_preci

In [ ]:
print("Evaluation metrics:")
for key, value in test_results.items():
    print(f"{key}: {value}")

Evaluation metrics:
eval_loss: 0.2078186422586441
eval_f1_micro: 0.8797741353619324
eval_precision_micro: 0.9077520634406862
eval_recall_micro: 0.8534692635423007
eval_f1_macro: 0.8808927265876648
eval_precision_macro: 0.9112146002177497
eval_recall_macro: 0.8534794575181756
eval_accuracy: 0.5846572855197458
eval_hamming_loss: 0.08698365864729914
eval_f1_label_0: 0.9394114252740912
eval_precision_label_0: 0.9476135040745053
eval_recall_label_0: 0.931350114416476
eval_support_label_0: 874
eval_f1_label_1: 0.9024691358024691
eval_precision_label_1: 0.938382541720154
eval_recall_label_1: 0.8692033293697978
eval_support_label_1: 841
eval_f1_label_2: 0.8155759870200108
eval_precision_label_2: 0.8240437158469945
eval_recall_label_2: 0.8072805139186295
eval_support_label_2: 934
eval_f1_label_3: 0.91005291005291
eval_precision_label_3: 0.9632
eval_recall_label_3: 0.8624641833810889
eval_support_label_3: 698
eval_f1_label_4: 0.8712871287128713
eval_precision_label_4: 0.8898876404494382
eval_rec

In [ ]:
print(type(model))
print(model.config.num_labels)
model.eval()


<class 'transformers.models.xlm_roberta.modeling_xlm_roberta.XLMRobertaForSequenceClassification'>
8


XLMRobertaForSequenceClassification(
  (classifier): XLMRobertaClassificationHead(
    (dense): Linear(in_features=768, out_features=768, bias=True)
    (dropout): Dropout(p=0.1, inplace=False)
    (out_proj): Linear(in_features=768, out_features=8, bias=True)
  )
  (roberta): XLMRobertaModel(
    (embeddings): XLMRobertaEmbeddings(
      (word_embeddings): Embedding(250002, 768, padding_idx=1)
      (token_type_embeddings): Embedding(1, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
      (position_embeddings): Embedding(514, 768, padding_idx=1)
    )
    (encoder): XLMRobertaEncoder(
      (layer): ModuleList(
        (0-11): 12 x XLMRobertaLayer(
          (attention): XLMRobertaAttention(
            (self): XLMRobertaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Li

In [ ]:
!pip install -q shap matplotlib seaborn scikit-learn

import torch
import numpy as np
import shap
import matplotlib.pyplot as plt
import seaborn as sns

torch.set_grad_enabled(False)

# IMPORTANT: model and tokenizer must already be loaded from your best checkpoint, e.g.:
# from transformers import AutoTokenizer, AutoModelForSequenceClassification
# MODEL_PATH = "/content/checkpoint-best"  # <-- your path
# tokenizer = AutoTokenizer.from_pretrained("Davlan/afro-xlmr-base")
# model = AutoModelForSequenceClassification.from_pretrained(MODEL_PATH)
model.eval()

print("Model type:", type(model))
print("Num labels:", model.config.num_labels)


In [ ]:
emotion_labels = [
     "sad", "happy", "anger",
    "fear", "disgust", "surprise", "contempt","neutral"
]

test_texts = [
    "እኔ አንዳንድ ግዜ እየተምታታብኝ ከዚህች ኢትዮጵያ ከምትባልና ሰላም አጥታ በጦርነትና በእርስ በእርስ ግጭት እየተናጠች መረጋጋት ባጣችና መሪ ባልተዋጣላት ሀገር መወለዴን በእጅጉ እያማረርኩ ቢሆንም ምሬቴንና ሀዘኔን የሚያስረሳ ውብ ሀገርማርና ወተት የሚፈስባት የምድር ባለፀጋ ለመሆናችን አርባ ምንጭን ሳይ ያሁሉ ምሬቴንና ብስጭቴን አርባ ምንጭን ሳይ ተካስኩኝ ከዚች ምድር መፈጠሬም እድለኛ መሆኔን መሰከርኩ ግን ለከብቶች ጥሩ እረኛ እንደሚያስፈል",   # joy + sadness
    "አንተ ባትወለድ፣ ሽጉጥህን እንደ ውሃ ባትጠጣ እኔን አልሆንም ነበር እኔ! መልካም ልደት አባቴማንነቴ!",           # anger + disgust
    "ሁለታቹም መጥፎ ናችሁ ያንቺው መጥፎነት ግን የከፋ ነው፡፡",                          # surprise + fear
    "እንኳን ደስ አላቹሁ መልካም ጋብቻ ይሁንላችሁ በአየር ሀይላችን በዓል ቀን የጋብቻ ቀለበት ስለአረጋቹ"                  # joy + neutral
]

print(f"🔍 Explaining {len(test_texts)} Amharic examples")

In [ ]:
import torch
torch.set_grad_enabled(False)
model.eval()   # make sure model is in eval mode

def predict_proba(text_list):
    enc = tokenizer(
        text_list,
        padding=True,
        truncation=True,
        max_length=128,
        return_tensors="pt"
    )
    with torch.no_grad():
        outputs = model(**enc)
        probs = torch.sigmoid(outputs.logits)   # (batch, num_labels)
    return probs.cpu().numpy()


In [ ]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()

def predict_proba(text_list):
    enc = tokenizer(
        text_list,
        padding=True,
        truncation=True,
        max_length=128,
        return_tensors="pt"
    )
    # move inputs to same device as model
    for k in enc:
        enc[k] = enc[k].to(device)

    with torch.no_grad():
        outputs = model(**enc)
        probs = torch.sigmoid(outputs.logits)  # (batch, num_labels)

    return probs.cpu().numpy()  # move back to CPU for SHAP/printing

def model_predict_proba_tokenized(input_ids_array, attention_mask_array=None):
    # SHAP's Text masker might provide NumPy arrays, convert them to PyTorch tensors
    input_ids_tensor = torch.tensor(input_ids_array, dtype=torch.long).to(device)
    inputs = {
        "input_ids": input_ids_tensor,
    }
    if attention_mask_array is not None:
        attention_mask_tensor = torch.tensor(attention_mask_array, dtype=torch.long).to(device)
        inputs["attention_mask"] = attention_mask_tensor

    with torch.no_grad():
        outputs = model(**inputs)
        probs = torch.sigmoid(outputs.logits)
    return probs.cpu().numpy()

In [ ]:
probs = predict_proba(test_texts) # Define probs before it's used
threshold = 0.3  # you can tune this, e.g. 0.4 or 0.6

for i, text in enumerate(test_texts):
    print("\nText:", text)
    active = []
    for label, p in zip(emotion_labels, probs[i]):
        if p >= threshold:
            active.append(f"{label} ({p:.3f})")
    if not active:
        # if nothing passes threshold, take top-1 label to avoid empty
        idx = probs[i].argmax()
        active = [f"{emotion_labels[idx]} ({probs[i][idx]:.3f})"]
    print("Predicted emotions:", ", ".join(active))


In [ ]:
probs = predict_proba(test_texts)
for i, text in enumerate(test_texts):
    print("\nText:", text)
    for label, p in zip(emotion_labels, probs[i]):
        print(f"  {label:9s}: {p:.3f}")


In [ ]:
!pip install -q shap matplotlib seaborn scikit-learn

import torch, numpy as np, shap, matplotlib.pyplot as plt

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()
torch.set_grad_enabled(False)

emotion_labels = [
    "sad", "happy", "anger", "fear",
    "disgust", "surprise", "contempt", "neutral"
]

test_texts = [
    "እኔ አንዳንድ ግዜ እየተምታታብኝ ከዚህች ኢትዮጵያ ...",   # long sad + anger
    "አንተ ባትወለድ፣ ሽጉጥህን እንደ ውሃ ...",           # birthday
    "ሁለታቹም መጥፎ ናችሁ ያንቺው መጥፎነት ግን የከፋ ነው፡፡",
    "እንኳን ደስ አላቹሁ መልካም ጋብቻ ይሁንላችሁ ..."
]

print("Explaining", len(test_texts), "Amharic examples")


In [ ]:
def predict_proba(text_list):
    # Ensure text_list is a list of strings.
    # SHAP's text masker can sometimes pass a single string or an array of strings.
    if isinstance(text_list, str):
        text_list = [text_list]
    elif isinstance(text_list, np.ndarray):
        text_list = text_list.tolist() # Convert numpy array of strings to python list of strings

    enc = tokenizer(
        text_list,
        padding=True,
        truncation=True,
        max_length=256,
        return_tensors="pt"
    )
    for k in enc:
        enc[k] = enc[k].to(device)

    with torch.no_grad():
        outputs = model(**enc)
        probs = torch.sigmoid(outputs.logits)   # (batch, 8)

    return probs.cpu().numpy()

In [ ]:
probs = predict_proba(test_texts)

for i, text in enumerate(test_texts):
    print("\nText:", text[:80], "...")
    for label, p in zip(emotion_labels, probs[i]):
        print(f"  {label:9s}: {p:.3f}")


In [ ]:
import torch, numpy as np, shap, matplotlib.pyplot as plt
torch.set_grad_enabled(False)
model.eval()


In [ ]:
!pip install lime
import torch
import numpy as np
from transformers import pipeline
from lime.lime_text import LimeTextExplainer
# Sample input for SHAP
sample_texts=[
    "ብአድን ካልጠፍ መቸም አማራ ሰላም ጠረነቱን አያሸንፍም በጣም ያሳዝናል",
    "ጠላት መነሀሪያ አድርጎታል እባካችሁ የሜድያ ባለቤቶች እውነት ስበአዋነት ይስማችሁ ከሆነ ድምፅ ሁኑን ለሚመለከተው አካል የተርፉት ቤተስቦቻችን ሂሳብ ማወራርጃ አይሁኑብን እባካችሁ ፍሲሎ ድምፅ ሁነን ሁሉም የምዕራብ ውሎ አገሮች ፍትህ ትኩርት ያሻል",
    "ማንው ፍኖን እሚበትነው ፍኖን እበትናለሁ የሚል አመራር የብአድን የብልጽግና የመንግስት አካላት ፍኖ ግባሩን ብሎቸው ብአደንን ግብርን በሉት",
    "ወይ አማራ ጠላቱ በዛ ጀግንነት ይሄን ያክል ያስፈትናል የአማራ ሆዳም አመራሮች እረፍ",
    "ቢአድን ጁታነው ከጁታው ተለይቶ አይታይም የአማራ ህዝብ ከፋኖ ጎንመቆም አለበት ድል ለፋኖ ለዘመነ ካሴ ያማራ ህዝብ ልብ በል አስተውል",
    "መንግስት ደግሞ ልክ እንደ ጌታቸው ምላሱ ምን ያስዋሸዋል ለምን ህዝብን ይሸውዳል",
    "ፍኖን በትነው ህዝብ ለዘር እኮን እንዳይተርፉ ይሙቱ ተብሎ ነው እንድ ህዝባቸውን እንዳይታደጉ የተደርጎት ምነው እነዚህ ፍኖን የሚበትኑት ባለስልጣናት ህግ ለእነሱ አይስራም እንደ ከአማራ ክልል አስተዳደር ጀምሮ ያሉትን",
    "በመጀመሪያ ብአደንን አስወግዱና የታስሩት ጀኔራሎቺ ይፍቱና ጦረነቱን ቀጥሉ",
    "በፍጥነት ድረሱላቸዉ በንደት ንፁሀኑን እንዳይጨርሷቸዉ ያረብ",
    "እነዚ ባአደኖች የአማራ ጥላት ናቸው መንግስት እርምጃ መውሥድ እይችልን እደ ባእደን ክመንግስት በላይ ነው እደ",
    "ፍትህ ለምእራብ ወሎ እባካችሁ ድረሡላቸዉ የበቀል እርምጃ እዳይወሡዱባችሁ ፍትህ ፍትህ ፍትህ ለሚመለከተዉ አካል",
    "ፍኖን የሚበትን አናቱ ይበተን! ፍኖን የበተነ ባለስልጣን አንድ ቀን ጊዜ መሰጠት የለበትም እርምጃ ማሰናበት ነው የምን ውይይት",
    "የአማራ ህዝብ ከህውሐት ቀጥሎ ትግሉ ከብአዴን ጋር ነው የሚሆነው አመኑኝ ምናለ በሉኝ",
    "አሁንም ስተት ከስሩ የአማራ አመራር ያናዳን ግን ስም ማጥፋት እንዳዪሆን የበተነው አካል ሊጠየቅ የገባን ያሳፍራን ያናዳን ወያኔን ለመጨረስ ከአማራ ጋር ሰዪጣንም ከተባበረ ጥሮ ነው እንኳን የአማራ ፋኖ"

]
labels=[[1,0,1,0,0,0,0,0],[1,0,1,1,0,0,0,0],[0,0,1,0,1,0,0,0],[0,0,1,0,1,0,0,0],
        [1,0,1,0,0,1,0,0],[0,0,1,0,0,1,0,0],[1,1,1,0,0,0,0,0],[0,0,1,1,1,0,0,0],
        [1,0,0,1,0,0,0,0],[0,0,1,0,1,1,0,0],[1,0,0,1,0,0,0,0],[0,0,1,0,1,0,1,0],
        [0,0,1,1,0,0,0,0],[0,0,1,0,1,0,1,0]]
# Wrap model + tokenizer
nlp_pipeline = pipeline("text-classification", model=model, tokenizer=tokenizer, return_all_scores=True)

def predict_proba(texts):
    inputs = tokenizer(texts, return_tensors="pt", padding=True, truncation=True)
    inputs = {k: v.to(model.device) for k, v in inputs.items()}  # move to model's device
    with torch.no_grad():
        outputs = model(**inputs)
        probs = torch.nn.functional.softmax(outputs.logits, dim=1)
    return probs.cpu().numpy()

num_labels = model.config.num_labels

# Create LIME explainer
explainer = LimeTextExplainer(class_names=[f"LABEL_{i}" for i in range(num_labels)])

# Explain each sample
for i, text in enumerate(sample_texts):
    print(f"\n🔍 Explaining sample {i+1}: {text}")
    exp = explainer.explain_instance(text, predict_proba, num_features=6, top_labels=1)
    exp.show_in_notebook(text=True)
